# Árvore de Atingibilidade e Árvore de Cobertura (Karp-Miller) em Redes de Petri
## Aplicação no Subsistema de Compressão e Cascata (`COMP-01` / `TK-AP`) — Estação de Hidrogênio (HRS)

**Disciplina:** ECAA08 • Engenharia de Controle e Automação & Matemática Discreta (UNIFEI)

Este notebook implementa:
1. O modelo formal $N = (P, T, Pre, Post, M_0)$ da Estação de Hidrogênio;
2. O **Algoritmo de Karp-Miller** (Árvore de Cobertura com símbolo $\omega$) detectando acúmulo ilimitado em `TK-AP` (Cenário A);
3. A **Síntese de Controle via Lugar Complementar** limitando `TK-AP` a $K_{\max} = 2$ lotes (Cenário B — Árvore de Atingibilidade finita);
4. Auditoria automática de propriedades (*Boundedness*, *Deadlock-freedom*, *Safety*) e visualização do grafo de estados.

### 1. Definição Formal da Rede de Petri e Algoritmo de Karp-Miller

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

OMEGA = float("inf")

def fmt_marcacao(m):
    return "(" + ", ".join("w" if x == OMEGA else str(int(x)) for x in m) + ")"

class RedeDePetri:
    def __init__(self, lugares, transicoes, pre, post, m0):
        self.lugares = lugares
        self.transicoes = transicoes
        self.pre = np.array(pre, dtype=float)
        self.post = np.array(post, dtype=float)
        self.m0 = tuple(float(x) for x in m0)

    def habilitada(self, m, j):
        return all(m[i] >= self.pre[i, j] for i in range(len(self.lugares)))

    def disparar(self, m, j):
        nova = []
        for i in range(len(self.lugares)):
            if m[i] == OMEGA:
                nova.append(OMEGA)
            else:
                nova.append(m[i] - self.pre[i, j] + self.post[i, j])
        return tuple(nova)

def construir_arvore_cobertura(rdp: RedeDePetri, max_nos: int = 200):
    """
    Constroi a Arvore de Cobertura (Karp-Miller) ou Arvore de Atingibilidade
    (quando a rede e limitada e nao gera simbolo omega).
    """
    G_estados = nx.DiGraph()
    G_estados.add_node(rdp.m0, label=fmt_marcacao(rdp.m0))

    # Fila armazena (marcacao_atual, lista_de_ancestrais_no_caminho)
    fila = [(rdp.m0, [rdp.m0])]
    visitados = {rdp.m0}

    while fila and len(G_estados.nodes) < max_nos:
        m_atual, ancestrais = fila.pop(0)
        for j, t_nome in enumerate(rdp.transicoes):
            if rdp.habilitada(m_atual, j):
                m_prox = list(rdp.disparar(m_atual, j))
                # Regra de substituicao omega de Karp-Miller
                for anc in ancestrais:
                    if all(m_prox[i] >= anc[i] for i in range(len(m_prox))) and tuple(m_prox) != anc:
                        for i in range(len(m_prox)):
                            if m_prox[i] > anc[i]:
                                m_prox[i] = OMEGA
                m_prox_t = tuple(m_prox)
                if m_prox_t not in G_estados:
                    G_estados.add_node(m_prox_t, label=fmt_marcacao(m_prox_t))
                G_estados.add_edge(m_atual, m_prox_t, label=t_nome)
                if m_prox_t not in visitados:
                    visitados.add(m_prox_t)
                    fila.append((m_prox_t, ancestrais + [m_prox_t]))
    return G_estados

### 2. Cenário A: Rede Sem Controle de Capacidade em `TK-AP` (Surgimento de $\omega$)

In [ ]:
lugares_A = ["P1_COMP_PRONTO", "P2_ADMISSAO_TKBP", "P3_COMPRESSAO", "P4_TKAP_LOTES", "P5_TRIP_ESD100"]
transicoes_A = ["T1_ABRIR_XV101", "T2_PARTIR_COMP01", "T3_FIM_COMP_TKAP", "T4_TRIP_SIS", "T5_RESET_SEGURO"]

pre_A = [
    [1, 0, 0, 0, 0],  # P1
    [0, 1, 0, 0, 0],  # P2
    [0, 0, 1, 1, 0],  # P3
    [0, 0, 0, 0, 0],  # P4
    [0, 0, 0, 0, 1],  # P5
]
post_A = [
    [0, 0, 1, 0, 1],  # P1
    [1, 0, 0, 0, 0],  # P2
    [0, 1, 0, 0, 0],  # P3
    [0, 0, 1, 0, 0],  # P4 (acúmulo sem drenagem)
    [0, 0, 0, 1, 0],  # P5
]
m0_A = [1, 0, 0, 0, 0]

rdp_A = RedeDePetri(lugares_A, transicoes_A, pre_A, post_A, m0_A)
G_A = construir_arvore_cobertura(rdp_A)

print("=== CENARIO A: ARVORE DE COBERTURA (SEM LIMITE EM TK-AP) ===")
for u, v, d in G_A.edges(data=True):
    print(f"{fmt_marcacao(u)} --[{d['label']}]--> {fmt_marcacao(v)}")

### 3. Cenário B: Síntese de Controle via Lugar Complementar (`P6_TKAP_LIVRE`, $K_{\max} = 2$)

In [ ]:
lugares_B = ["P1_COMP_PRONTO", "P2_ADMISSAO_TKBP", "P3_COMPRESSAO", "P4_TKAP_LOTES", "P5_TRIP_ESD100", "P6_TKAP_LIVRE"]
transicoes_B = ["T1_ABRIR_XV101", "T2_PARTIR_COMP01", "T3_FIM_COMP_TKAP", "T4_TRIP_SIS", "T5_RESET_SEGURO", "T6_DISPENSAR_H2"]

pre_B = [
    [1, 0, 0, 0, 0, 0],  # P1
    [0, 1, 0, 0, 0, 0],  # P2
    [0, 0, 1, 1, 0, 0],  # P3
    [0, 0, 0, 0, 0, 1],  # P4 (T6 consome 1 lote para abastecer FCEV)
    [0, 0, 0, 0, 1, 0],  # P5
    [0, 0, 1, 0, 0, 0],  # P6 (T3 requer espaco livre em TK-AP)
]
post_B = [
    [0, 0, 1, 0, 1, 0],  # P1
    [1, 0, 0, 0, 0, 0],  # P2
    [0, 1, 0, 0, 0, 0],  # P3
    [0, 0, 1, 0, 0, 0],  # P4
    [0, 0, 0, 1, 0, 0],  # P5
    [0, 0, 0, 0, 0, 1],  # P6 (T6 devolve 1 espaco livre em TK-AP)
]
m0_B = [1, 0, 0, 0, 0, 2]  # Capacidade maxima de 2 lotes em TK-AP

rdp_B = RedeDePetri(lugares_B, transicoes_B, pre_B, post_B, m0_B)
G_B = construir_arvore_cobertura(rdp_B)

def auditar_propriedades_rdp(rdp: RedeDePetri, grafo_estados: nx.DiGraph, nome: str):
    estados = list(grafo_estados.nodes)
    ilimitada = any(OMEGA in e for e in estados)
    deadlocks = [fmt_marcacao(e) for e in estados if grafo_estados.out_degree(e) == 0]
    exclusao_ok = not any(e[2] > 0 and e[4] > 0 for e in estados)  # P3 (Compressao) e P5 (Trip ESD) nunca juntos
    print(f"=== AUDITORIA FORMAL ({nome}) ===")
    print(f"Total de Estados Alcançáveis: {len(estados)}")
    print(f"Rede Limitada (Bounded)?      {not ilimitada}")
    print(f"Deadlocks Detectados:         {len(deadlocks)} {deadlocks}")
    print(f"Exclusão Mútua COMP vs ESD:   {exclusao_ok}")

auditar_propriedades_rdp(rdp_A, G_A, "Cenario A - Sem Controle")
print()
auditar_propriedades_rdp(rdp_B, G_B, "Cenario B - Com Lugar Complementar")

### 4. Visualização Gráfica do Grafo de Atingibilidade (Cenário B)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
pos = nx.spring_layout(G_B, seed=42, k=1.4)
node_colors = ["gold" if n == rdp_B.m0 else ("salmon" if n[4] > 0 else "lightblue") for n in G_B.nodes]

nx.draw_networkx_nodes(G_B, pos, ax=ax, node_color=node_colors, node_size=2200, edgecolors="black")
labels = {n: fmt_marcacao(n) for n in G_B.nodes}
nx.draw_networkx_labels(G_B, pos, labels=labels, ax=ax, font_size=7.5, font_weight="bold")
nx.draw_networkx_edges(G_B, pos, ax=ax, edge_color="dimgray", arrows=True, arrowsize=15, connectionstyle="arc3,rad=0.12")

edge_labels = {(u, v): d["label"].split("_")[0] for u, v, d in G_B.edges(data=True)}
nx.draw_networkx_edge_labels(G_B, pos, edge_labels=edge_labels, ax=ax, font_size=7.5)

ax.set_title("Grafo de Atingibilidade Controlado — HRS (Dourado=M0, Vermelho=Trip ESD-100)", fontsize=12)
ax.axis("off")
plt.tight_layout()
plt.show()